In [1]:
!pip install -q -U transformers accelerate datasets sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 86.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 84.8 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.

In [2]:
import os
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)

from datasets import Dataset

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(DEVICE)

cuda


In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
label2id = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

id2label = {v:k for k,v in label2id.items()}

train["label"] = train["answer"].map(label2id)

In [7]:
train_df, valid_df = train_test_split(
    train,
    test_size=0.1,
    stratify=train["label"],
    random_state=SEED
)

print(train_df.shape)
print(valid_df.shape)

(1800, 9)
(200, 9)


In [8]:
MODEL_NAME = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [18]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test.reset_index(drop=True))

In [30]:
MAX_LENGTH = 256

option_to_text = ["A", "B", "C", "D", "E"]

def preprocess_function(examples):
    first_sentences = []
    second_sentences = []

    for prompt in examples["prompt"]:
        first_sentences.extend([prompt] * 5)

    for i in range(len(examples["prompt"])):
        second_sentences.extend([
            examples["A"][i],
            examples["B"][i],
            examples["C"][i],
            examples["D"][i],
            examples["E"][i]
        ])

    tokenized = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )

    result = {
        k: [v[i:i+5] for i in range(0, len(v), 5)]
        for k, v in tokenized.items()
    }

    if "label" in examples:
        result["labels"] = examples["label"]

    return result

In [31]:
encoded_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

encoded_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

encoded_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [32]:
print(encoded_train[0].keys())

print(len(encoded_train[0]["input_ids"]))

print(len(encoded_train[0]["input_ids"][0]))

dict_keys(['input_ids', 'attention_mask', 'labels'])
5
256


In [33]:
model = AutoModelForMultipleChoice.from_pretrained(
    MODEL_NAME
)

model.to(DEVICE)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForMultipleChoice LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ModernBertForMultipleChoice(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
        (attn_n

In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

In [35]:
training_args = TrainingArguments(
    output_dir="./modernbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train,
    eval_dataset=encoded_valid,
    compute_metrics=compute_metrics
)

In [37]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy
1,0.881188,0.345921,0.990000
2,0.033480,0.018987,1.000000
3,0.000064,0.000063,1.000000
4,0.000020,0.000042,1.000000
5,0.000012,0.000039,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=565, training_loss=0.3502335806648696, metrics={'train_runtime': 1533.8109, 'train_samples_per_second': 5.868, 'train_steps_per_second': 0.368, 'total_flos': 7666998382080000.0, 'train_loss': 0.3502335806648696, 'epoch': 5.0})

In [38]:
test_predictions = trainer.predict(encoded_test)

In [43]:
# Get logits
test_predictions = trainer.predict(encoded_test)
logits = test_predictions.predictions

In [47]:
import numpy as np

id2label = {
    0: "A",
    1: "B",
    2: "C",
    3: "D",
    4: "E"
}

top3 = np.argsort(logits, axis=1)[:, ::-1][:, :3]

predictions = [
    " ".join(id2label[i] for i in row)
    for row in top3
]

submission["Prediction"] = predictions

predictions[:10]

['A B C',
 'B C A',
 'B D C',
 'E A D',
 'C A D',
 'D C B',
 'E D C',
 'B C A',
 'C D A',
 'B E C']

In [48]:
submission = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"
)

submission["Prediction"] = predictions

submission.head()

,ID,Prediction
0,1,A B C
1,2,B C A
2,3,B D C
3,4,E A D
4,5,C A D


In [51]:
submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv(
    "submission.csv",
    index=False
)

submission.head()

,id,prediction
0,1,A B C
1,2,B C A
2,3,B D C
3,4,E A D
4,5,C A D


In [52]:
import os
import shutil
import pandas as pd

assert os.path.exists("submission.csv"), "submission.csv not found!"

submission = pd.read_csv("submission.csv")
print(submission.head())

os.makedirs("/kaggle/working/final_submission", exist_ok=True)
shutil.copy("submission.csv", "/kaggle/working/final_submission/submission.csv")

print("Saved successfully!")

   id prediction
0   1      A B C
1   2      B C A
2   3      B D C
3   4      E A D
4   5      C A D
Saved successfully!


In [53]:
import os

print(os.listdir("/kaggle/working"))
print(os.listdir("/kaggle/working/final_submission"))

['.virtual_documents', 'modernbert', 'submission.csv', 'final_submission']
['submission.csv']


In [55]:
import os

print(os.getcwd())
print(os.listdir())

/kaggle/working
['.virtual_documents', 'modernbert', 'submission.csv', 'final_submission']


In [56]:
submission.to_csv("submission.csv", index=False)
print("Saved!")

Saved!


In [57]:
import os
print(os.path.exists("submission.csv"))

True
